In [13]:
from curses import raw
import os
import re
import json
import numpy as np
import pandas as pd
import mne

# =========================
# CONFIG
# =========================

DATA_ROOT = "data/raw"              # root folder containing chb01, chb02, ...
OUTPUT_ROOT = "data/processed"      # where processed arrays + metadata will be saved

PATIENT_IDS = ["chb01","chb02","chb06","chb08","chb10","chb12","chb15","chb18"]   # start with clean patients first

WINDOW_SIZE_SEC = 5.0               # each sample window length
STRIDE_SEC = 2                    
OVERLAP_LABEL_THRESHOLD = 0.2       # label seizure if >= 20% of window overlaps seizure interval

# Stable bipolar channel set for clean patients
TARGET_CHANNELS = [
    "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
    "FZ-CZ", "CZ-PZ"
]
# =========================
# HELPERS
# =========================

def normalize_channel_name(ch: str):
    ch = ch.strip().upper()
    ch = ch.replace("EEG ", "")
    ch = ch.replace(" ", "")
    ch = ch.replace("--", "-")

    parts = ch.split("-")
    if len(parts) >= 2 and parts[-1].isdigit():
        ch = "-".join(parts[:-1])

    if ch in {".", "", "-"}:
        return None

    return ch


def parse_patient_summary(data_root: str, patient_id: str, save_json: bool = True):
    """
    Returns:
        {
            "chb01_01.edf": [],
            "chb01_03.edf": [(2996, 3036)],
            ...
        }
    """
    patient_path = os.path.join(data_root, patient_id)

    if not os.path.exists(patient_path):
        raise FileNotFoundError(f"Patient folder not found: {patient_path}")

    summary_file = None
    for f in os.listdir(patient_path):
        if "summary" in f.lower():
            summary_file = os.path.join(patient_path, f)
            break

    if summary_file is None:
        raise FileNotFoundError(f"No summary file found inside {patient_path}")

    with open(summary_file, "r") as f:
        lines = [line.strip() for line in f.readlines()]

    seizure_map = {}

    current_file = None
    current_intervals = []
    pending_start = None

    file_pat = re.compile(r"^File Name:\s*(.+)$")
    seizure_start_pat = re.compile(
        r"^Seizure(?:\s+\d+)?\s+Start Time:\s*(\d+)\s+seconds$",
        re.IGNORECASE
    )
    seizure_end_pat = re.compile(
        r"^Seizure(?:\s+\d+)?\s+End Time:\s*(\d+)\s+seconds$",
        re.IGNORECASE
    )

    for line in lines:
        m_file = file_pat.match(line)
        if m_file:
            if current_file is not None:
                seizure_map[current_file] = current_intervals

            current_file = m_file.group(1).strip()
            current_intervals = []
            pending_start = None
            continue

        m_start = seizure_start_pat.match(line)
        if m_start:
            pending_start = int(m_start.group(1))
            continue

        m_end = seizure_end_pat.match(line)
        if m_end and pending_start is not None:
            end_time = int(m_end.group(1))
            current_intervals.append((pending_start, end_time))
            pending_start = None

    if current_file is not None:
        seizure_map[current_file] = current_intervals

    if save_json:
        out_dir = os.path.join(OUTPUT_ROOT, "metadata")
        os.makedirs(out_dir, exist_ok=True)

        save_path = os.path.join(out_dir, f"{patient_id}_seizure_map.json")
        json_safe = {
            fname: [[s, e] for s, e in intervals]
            for fname, intervals in seizure_map.items()
        }
        with open(save_path, "w") as f:
            json.dump(json_safe, f, indent=2)

    return seizure_map


def load_edf_with_target_channels(edf_path, target_channels):
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)

    original_names = raw.ch_names
    target_norm = [normalize_channel_name(ch) for ch in target_channels]

    # 3) mapping block
    norm_to_orig = {}
    for orig in original_names:
        norm = normalize_channel_name(orig)
        if norm is None:
            continue
        if norm not in norm_to_orig:
            norm_to_orig[norm] = orig

    # 4) debug prints + missing check
    print("\nEDF:", os.path.basename(edf_path))
    print("Original names:", original_names)
    print("Normalized available:", sorted(norm_to_orig.keys()))
    print("Target normalized:", target_norm)

    missing = [ch for ch in target_norm if ch not in norm_to_orig]
    if missing:
        raise ValueError(f"Missing target channels: {missing}")

    ordered_orig_names = [norm_to_orig[ch] for ch in target_norm]
    raw.pick(ordered_orig_names)

    data = raw.get_data().astype(np.float64)
    sfreq = raw.info["sfreq"]

    print("FINAL SHAPE:", data.shape)
    print("CHANNEL ORDER:", ordered_orig_names)

    return data, sfreq, ordered_orig_names

def generate_windows(data: np.ndarray, sfreq: float, window_size_sec: float, stride_sec: float):
    """
    data shape: (n_channels, n_timepoints)

    Returns:
        X_windows: shape (n_windows, n_channels, n_window_timepoints)
        window_times: list of tuples [(start_sec, end_sec), ...]
    """
    n_channels, n_timepoints = data.shape

    window_size = int(window_size_sec * sfreq)
    stride = int(stride_sec * sfreq)

    if n_timepoints < window_size:
        return np.empty((0, n_channels, window_size), dtype=np.float32), []

    windows = []
    window_times = []

    start_idx = 0
    while start_idx + window_size <= n_timepoints:
        end_idx = start_idx + window_size

        window = data[:, start_idx:end_idx]
        windows.append(window)

        start_sec = start_idx / sfreq
        end_sec = end_idx / sfreq
        window_times.append((start_sec, end_sec))

        start_idx += stride

    X_windows = np.stack(windows).astype(np.float32)
    return X_windows, window_times


def compute_overlap(a_start, a_end, b_start, b_end):
    """
    Overlap duration between [a_start, a_end] and [b_start, b_end]
    """
    left = max(a_start, b_start)
    right = min(a_end, b_end)
    return max(0.0, right - left)


def label_windows(window_times, seizure_intervals, overlap_threshold=0.5):
    """
    window_times: [(start_sec, end_sec), ...]
    seizure_intervals: [(start_sec, end_sec), ...]

    Returns:
        y: np.ndarray shape (n_windows,)
    """
    y = np.zeros(len(window_times), dtype=np.int64)

    for i, (w_start, w_end) in enumerate(window_times):
        w_len = w_end - w_start

        for s_start, s_end in seizure_intervals:
            overlap = compute_overlap(w_start, w_end, s_start, s_end)
            overlap_ratio = overlap / w_len if w_len > 0 else 0.0

            if overlap_ratio >= overlap_threshold:
                y[i] = 1
                break

    return y


def save_file_outputs(patient_id, edf_filename, X, y):
    """
    Save one X.npy and one y.npy per EDF file.
    Returns saved paths.
    """
    patient_out_dir = os.path.join(OUTPUT_ROOT, "windows", patient_id)
    os.makedirs(patient_out_dir, exist_ok=True)

    base = os.path.splitext(edf_filename)[0]
    x_path = os.path.join(patient_out_dir, f"{base}_X.npy")
    y_path = os.path.join(patient_out_dir, f"{base}_y.npy")

    np.save(x_path, X)
    np.save(y_path, y)

    return x_path, y_path


def process_one_edf(patient_id, edf_filename, seizure_intervals, target_channels):
    """
    Full pipeline for one EDF file.
    Returns:
        metadata_rows: list[dict]
        file_summary: dict
    """
    edf_path = os.path.join(DATA_ROOT, patient_id, edf_filename)

    if not os.path.exists(edf_path):
        raise FileNotFoundError(f"EDF not found: {edf_path}")

    data, sfreq, used_channels = load_edf_with_target_channels(edf_path, target_channels)

    print(data.shape)
    print(used_channels)
    data = bandpass_filter(data, sfreq, l_freq=0.5, h_freq=40.0)
    X, window_times = generate_windows(
        data=data,
        sfreq=sfreq,
        window_size_sec=WINDOW_SIZE_SEC,
        stride_sec=STRIDE_SEC
    )
    y = label_windows(
        window_times=window_times,
        seizure_intervals=seizure_intervals,
        overlap_threshold=OVERLAP_LABEL_THRESHOLD
    )

    x_path, y_path = save_file_outputs(patient_id, edf_filename, X, y)

    metadata_rows = []
    for idx, ((start_sec, end_sec), label) in enumerate(zip(window_times, y)):
        metadata_rows.append({
            "patient_id": patient_id,
            "file_name": edf_filename,
            "window_idx": idx,
            "start_sec": float(start_sec),
            "end_sec": float(end_sec),
            "label": int(label),
            "x_path": x_path,
            "y_path": y_path,
            "sfreq": sfreq,
            "n_channels": X.shape[1] if X.size > 0 else len(target_channels),
            "n_timepoints": X.shape[2] if X.size > 0 else int(WINDOW_SIZE_SEC * sfreq)
        })

    file_summary = {
        "patient_id": patient_id,
        "file_name": edf_filename,
        "n_windows": int(len(y)),
        "n_positive_windows": int(y.sum()),
        "n_negative_windows": int((y == 0).sum()),
        "sfreq": sfreq,
        "used_channels": used_channels
    }

    return metadata_rows, file_summary


def process_patient(patient_id, target_channels):
    """
    Process all EDF files listed in that patient's summary.
    Returns:
        patient_index_df
        patient_file_summary_df
        patient_skip_df
    """
    seizure_map = parse_patient_summary(DATA_ROOT, patient_id, save_json=True)

    all_rows = []
    file_summaries = []
    skipped = []

    for edf_filename, seizure_intervals in seizure_map.items():
        try:
            metadata_rows, file_summary = process_one_edf(
                patient_id=patient_id,
                edf_filename=edf_filename,
                seizure_intervals=seizure_intervals,
                target_channels=target_channels
            )
            all_rows.extend(metadata_rows)
            file_summaries.append(file_summary)

            print(
                f"[OK] {patient_id}/{edf_filename} | "
                f"windows={file_summary['n_windows']} | "
                f"positive={file_summary['n_positive_windows']}"
            )

        except Exception as e:
            skipped.append({
                "patient_id": patient_id,
                "file_name": edf_filename,
                "reason": str(e)
            })
            print(f"[SKIP] {patient_id}/{edf_filename} -> {e}")

    patient_index_df = pd.DataFrame(all_rows)
    patient_file_summary_df = pd.DataFrame(file_summaries)
    patient_skip_df = pd.DataFrame(skipped)

    return patient_index_df, patient_file_summary_df, patient_skip_df


def save_global_artifacts(index_df, file_summary_df, skip_df, target_channels):
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_ROOT, "metadata"), exist_ok=True)

    index_csv = os.path.join(OUTPUT_ROOT, "sample_index.csv")
    files_csv = os.path.join(OUTPUT_ROOT, "file_summary.csv")
    skip_csv = os.path.join(OUTPUT_ROOT, "skipped_files.csv")
    channels_json = os.path.join(OUTPUT_ROOT, "channels.json")
    config_json = os.path.join(OUTPUT_ROOT, "config.json")

    index_df.to_csv(index_csv, index=False)
    file_summary_df.to_csv(files_csv, index=False)
    skip_df.to_csv(skip_csv, index=False)

    with open(channels_json, "w") as f:
        json.dump(target_channels, f, indent=2)

    config = {
        "data_root": DATA_ROOT,
        "output_root": OUTPUT_ROOT,
        "patient_ids": PATIENT_IDS,
        "window_size_sec": WINDOW_SIZE_SEC,
        "stride_sec": STRIDE_SEC,
        "overlap_label_threshold": OVERLAP_LABEL_THRESHOLD,
        "target_channels": TARGET_CHANNELS
    }
    with open(config_json, "w") as f:
        json.dump(config, f, indent=2)

    print("\nSaved:")
    print(index_csv)
    print(files_csv)
    print(skip_csv)
    print(channels_json)
    print(config_json)

def bandpass_filter(data, sfreq, l_freq=0.5, h_freq=40.0):
    """
    Apply bandpass filter to EEG data
    data: (n_channels, n_timepoints)
    """
    return mne.filter.filter_data(
        data,
        sfreq=sfreq,
        l_freq=l_freq,
        h_freq=h_freq,
        method='fir',
        verbose=False
    ).astype(np.float64)


def main():
    all_index_dfs = []
    all_file_summary_dfs = []
    all_skip_dfs = []

    for patient_id in PATIENT_IDS:
        print(f"\n========== Processing {patient_id} ==========")
        index_df, file_summary_df, skip_df = process_patient(patient_id, TARGET_CHANNELS)

        all_index_dfs.append(index_df)
        all_file_summary_dfs.append(file_summary_df)
        all_skip_dfs.append(skip_df)

    final_index_df = pd.concat(all_index_dfs, ignore_index=True) if all_index_dfs else pd.DataFrame()
    final_file_summary_df = pd.concat(all_file_summary_dfs, ignore_index=True) if all_file_summary_dfs else pd.DataFrame()
    final_skip_df = pd.concat(all_skip_dfs, ignore_index=True) if all_skip_dfs else pd.DataFrame()

    save_global_artifacts(
        index_df=final_index_df,
        file_summary_df=final_file_summary_df,
        skip_df=final_skip_df,
        target_channels=TARGET_CHANNELS
    )

    print("\n========== DONE ==========")
    print(f"Total window rows: {len(final_index_df)}")
    if not final_index_df.empty:
        print(f"Positive windows: {int(final_index_df['label'].sum())}")
        print(f"Negative windows: {int((final_index_df['label'] == 0).sum())}")
        


if __name__ == "__main__":
    main()


========== Processing chb01 ==========

EDF: chb01_01.edf
Original names: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Normalized available: ['C3-P3', 'C4-P4', 'CZ-PZ', 'F3-C3', 'F4-C4', 'F7-T7', 'F8-T8', 'FP1-F3', 'FP1-F7', 'FP2-F4', 'FP2-F8', 'FT10-T8', 'FT9-FT10', 'FZ-CZ', 'P3-O1', 'P4-O2', 'P7-O1', 'P7-T7', 'P8-O2', 'T7-FT9', 'T7-P7', 'T8-P8']
Target normalized: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
FINAL SHAPE: (18, 921600)
CHANNEL ORDER: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
(18, 921600)
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3

/var/folders/8g/ttg2t0td2blcgq4cm80ph_b80000gn/T/ipykernel_39727/3662587275.py:133: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


[OK] chb01/chb01_01.edf | windows=1798 | positive=0

EDF: chb01_02.edf
Original names: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Normalized available: ['C3-P3', 'C4-P4', 'CZ-PZ', 'F3-C3', 'F4-C4', 'F7-T7', 'F8-T8', 'FP1-F3', 'FP1-F7', 'FP2-F4', 'FP2-F8', 'FT10-T8', 'FT9-FT10', 'FZ-CZ', 'P3-O1', 'P4-O2', 'P7-O1', 'P7-T7', 'P8-O2', 'T7-FT9', 'T7-P7', 'T8-P8']
Target normalized: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
FINAL SHAPE: (18, 921600)
CHANNEL ORDER: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
(18, 921600)
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3

/var/folders/8g/ttg2t0td2blcgq4cm80ph_b80000gn/T/ipykernel_39727/3662587275.py:133: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


[OK] chb01/chb01_02.edf | windows=1798 | positive=0

EDF: chb01_03.edf
Original names: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Normalized available: ['C3-P3', 'C4-P4', 'CZ-PZ', 'F3-C3', 'F4-C4', 'F7-T7', 'F8-T8', 'FP1-F3', 'FP1-F7', 'FP2-F4', 'FP2-F8', 'FT10-T8', 'FT9-FT10', 'FZ-CZ', 'P3-O1', 'P4-O2', 'P7-O1', 'P7-T7', 'P8-O2', 'T7-FT9', 'T7-P7', 'T8-P8']
Target normalized: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
FINAL SHAPE: (18, 921600)
CHANNEL ORDER: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
(18, 921600)
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3

...

[OK] chb18/chb18_34.edf | windows=1798 | positive=0

EDF: chb18_35.edf
Original names: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', '.-0', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', '.-1', 'FZ-CZ', 'CZ-PZ', '.-2', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', '.-3', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', '.-4', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Normalized available: ['C3-P3', 'C4-P4', 'CZ-PZ', 'F3-C3', 'F4-C4', 'F7-T7', 'F8-T8', 'FP1-F3', 'FP1-F7', 'FP2-F4', 'FP2-F8', 'FT10-T8', 'FT9-FT10', 'FZ-CZ', 'P3-O1', 'P4-O2', 'P7-O1', 'P7-T7', 'P8-O2', 'T7-FT9', 'T7-P7', 'T8-P8']
Target normalized: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
FINAL SHAPE: (18, 921600)
CHANNEL ORDER: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
(18, 921600)
['FP1-F7', 'F7-T

/var/folders/8g/ttg2t0td2blcgq4cm80ph_b80000gn/T/ipykernel_39727/3662587275.py:133: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8', '.'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


[OK] chb18/chb18_35.edf | windows=1798 | positive=36

EDF: chb18_36.edf
Original names: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', '.-0', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', '.-1', 'FZ-CZ', 'CZ-PZ', '.-2', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', '.-3', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', '.-4', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Normalized available: ['C3-P3', 'C4-P4', 'CZ-PZ', 'F3-C3', 'F4-C4', 'F7-T7', 'F8-T8', 'FP1-F3', 'FP1-F7', 'FP2-F4', 'FP2-F8', 'FT10-T8', 'FT9-FT10', 'FZ-CZ', 'P3-O1', 'P4-O2', 'P7-O1', 'P7-T7', 'P8-O2', 'T7-FT9', 'T7-P7', 'T8-P8']
Target normalized: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
FINAL SHAPE: (18, 921600)
CHANNEL ORDER: ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
(18, 921600)
['FP1-F7', 'F7-

/var/folders/8g/ttg2t0td2blcgq4cm80ph_b80000gn/T/ipykernel_39727/3662587275.py:133: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8', '.'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


[OK] chb18/chb18_36.edf | windows=1798 | positive=25

Saved:
data/processed/sample_index.csv
data/processed/file_summary.csv
data/processed/skipped_files.csv
data/processed/channels.json
data/processed/config.json

========== DONE ==========
Total window rows: 555579
Positive windows: 2885
Negative windows: 552694


In [10]:
print(data.shape)

NameError: name 'data' is not defined

In [11]:
print(ordered_orig_names)

NameError: name 'ordered_orig_names' is not defined